# ODOT Standard Objects

In [1]:
from ODOT import BrRBridge

In [2]:
my_bridge = BrRBridge('6006183')
print(my_bridge)

BrRBridge('6006183', 'MUS-00555-06.680')
  Controlling Rating:
    Inventory RF: 0.6596606373786926
    Operating RF: 0.8551156520843506
    Legal RF:     None
    Permit RF:    None
    Vehicle:      HL-93 (US)
    Method:       None
    Limit State:  STRENGTH-I Steel Flexure Stress
  Member Capacities: 0 points
  Vehicle Loads:     11 vehicles


In [3]:
my_bridge.fetch_design_info()

In [4]:
my_bridge.display_design_summary()


Design Summary for 6006183:
Member               | Depth (mm) | Web (mm) | Top Flg  | Comp?
-----------------------------------------------------------------
Girder 1             | 1981.20    | 11.11    | N/A      | No
Girder 1             | 1981.20    | 11.11    | N/A      | No
Girder 1             | 1981.20    | 11.11    | N/A      | No
Unit1 Stringer1      | N/A        | N/A      | N/A      | No
Unit1 Stringer2      | N/A        | N/A      | N/A      | No
Unit2 Stringer1      | N/A        | N/A      | N/A      | No
Unit2 Stringer2      | N/A        | N/A      | N/A      | No
Unit3 Stringer1      | N/A        | N/A      | N/A      | No
Unit3 Stringer2      | N/A        | N/A      | N/A      | No
Floorbeam1           | N/A        | N/A      | N/A      | No
Floorbeam2           | 402.59     | 7.59     | 177.60   | No
Floorbeam3           | 402.59     | 7.59     | 177.60   | No
Floorbeam4           | 402.59     | 7.59     | 177.60   | No
Floorbeam5           | 402.59     | 7.59     | 1

# TIMs

In [6]:
from ODOT import TIMSBridge, TIMSProject, TIMSRoad, TIMSWaterway

In [7]:
# ---------------------------------------------------------
# 1. TIMSBridge: Deep dive into a specific structure
# ---------------------------------------------------------
print("--- Bridge Deep Dive ---")
# Instantiate by SFN (Structure File Number)
bridge = TIMSBridge("2102226")

print(f"Location: {bridge.county} County, District {bridge.district}")
print(f"Carries {bridge.facility_carried} over {bridge.feature_intersected}")
print(f"Built in {bridge.year_built}, Sufficiency Rating: {bridge.sufficiency_rating}")

# Built-in logic checks
if bridge.is_structurally_deficient:
    print("⚠️ Warning: Bridge is structurally deficient.")
    print(f"Condition Ratings: {bridge.condition_ratings}")

if bridge.is_scour_critical:
    print("⚠️ Warning: Bridge is scour critical.")

# You can also do spatial/attribute searches without instantiating a single bridge
delaware_bridges = TIMSBridge.search_by_county("DEL")
print(f"\nFound {len(delaware_bridges)} bridges in Delaware County.")

--- Bridge Deep Dive ---
Location: DEL County, District 06
Carries I-71 NB over 3.55 MI N OF FRANKLIN CO
Built in 1959, Sufficiency Rating: 89.9

Found 581 bridges in Delaware County.


In [8]:
# ---------------------------------------------------------
# 2. TIMSProject: District Work Plans & Construction
# ---------------------------------------------------------
print("\n--- Project Scanning ---")
# Find all active/planned projects in District 6
d6_projects = TIMSProject.search_by_district("06")
print(f"Found {len(d6_projects)} projects in District 06.")

# Check for projects happening near our bridge
if bridge.lon and bridge.lat:
    nearby_projects = TIMSProject.search_near(
        lon=bridge.lon, 
        lat=bridge.lat, 
        radius_miles=2.0
    )
    for proj in nearby_projects:
        print(f"Nearby PID: {proj.get('PID_NBR')} - {proj.get('PROJECT_NME')}")


--- Project Scanning ---
Found 0 projects in District 06.


In [9]:
# ---------------------------------------------------------
# 3. TIMSRoad & TIMSWaterway: Environmental & Network Data
# ---------------------------------------------------------
print("\n--- Network & Environmental Constraints ---")
if bridge.lon and bridge.lat:
    # Check if this bridge sits on the NHS (National Highway System)
    nhs_routes = TIMSRoad.nhs_routes_near(bridge.lon, bridge.lat, radius_miles=0.1)
    if nhs_routes:
        print("🛣️ Bridge is on or adjacent to an NHS route.")

    # Check for environmental constraints before planning work
    mussel_streams = TIMSWaterway.mussel_streams_near(bridge.lon, bridge.lat, radius_miles=1.0)
    if mussel_streams:
        print(f"🦪 Found {len(mussel_streams)} protected mussel stream(s) within 1 mile.")
        
    scenic_rivers = TIMSWaterway.scenic_rivers_near(bridge.lon, bridge.lat, radius_miles=1.0)
    if scenic_rivers:
        print("🏞️ Warning: Scenic river restrictions may apply.")


--- Network & Environmental Constraints ---
🦪 Found 1 protected mussel stream(s) within 1 mile.
